In [2]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [33]:
from groq import Groq

client = Groq(api_key="Your Key")  

response = client.chat.completions.create(
    model    = "llama-3.3-70b-versatile",
    messages = [{"role": "user", "content": "Say hello in one sentence"}]
)

print(response.choices[0].message.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [34]:
import pandas as pd
import joblib
import numpy as np




df = pd.read_csv('RideFlow_Processed_Data.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFeedback sample:")
print(df['feedback_text'].value_counts())

Shape: (50000, 30)

Columns: ['ride_id', 'timestamp', 'pickup_zone', 'drop_zone', 'pickup_lat', 'pickup_long', 'drop_lat', 'drop_long', 'driver_id', 'customer_id', 'fare_price', 'surge_multiplier', 'driver_rating', 'customer_rating', 'estimated_eta_min', 'actual_eta_min', 'ride_status', 'traffic_level', 'weather', 'driver_active', 'feedback_text', 'year', 'month', 'day', 'hour', 'minute', 'day_of_week', 'is_weekend', 'eta_error', 'pickup_cluster']

Feedback sample:
feedback_text
Vehicle was not clean        10760
Driver cancelled suddenly    10194
Driver was polite            10095
Good experience               9658
Ride was delayed              9293
Name: count, dtype: int64


In [24]:
#AS data was not enough i have added a synthenic feedback data to make the Bert work more effciently

In [35]:
import random

feedback_templates = {
    "Positive": [
        "The driver was very polite and helpful",
        "Smooth ride, reached on time, great experience",
        "Driver followed the route perfectly, very satisfied",
        "Very clean vehicle and friendly driver",
        "Excellent service, will use again",
        "Driver was professional and courteous",
        "Comfortable ride, no issues at all"
    ],
    "Neutral": [
        "Ride was okay, nothing special",
        "Average experience, driver was quiet",
        "Reached destination but took longer than expected",
        "Neither good nor bad, just a normal ride",
        "Driver was fine, vehicle was average"
    ],
    "Negative": [
        "Driver was rude and unprofessional",
        "Vehicle was very dirty and uncomfortable",
        "Driver cancelled last minute, very inconvenient",
        "Ride was delayed by 20 minutes, unacceptable",
        "Driver took wrong route and overcharged",
        "Very bad experience, driver was aggressive",
        "AC was not working, driver was unhelpful"
    ]
}

# Assign sentiment label and matching feedback
sentiments = []
feedbacks  = []

for _ in range(len(df)):
    sentiment = random.choices(
        ['Positive', 'Neutral', 'Negative'],
        weights=[0.5, 0.2, 0.3]
    )[0]
    sentiments.append(sentiment)
    feedbacks.append(random.choice(feedback_templates[sentiment]))

df['feedback_text']      = feedbacks
df['sentiment_label']    = sentiments

print(df['sentiment_label'].value_counts())
print("\nSample:")
print(df[['feedback_text', 'sentiment_label']].head(10))

sentiment_label
Positive    25071
Negative    15010
Neutral      9919
Name: count, dtype: int64

Sample:
                                       feedback_text sentiment_label
0           AC was not working, driver was unhelpful        Negative
1                     Ride was okay, nothing special         Neutral
2            Driver took wrong route and overcharged        Negative
3                 Driver was rude and unprofessional        Negative
4              Driver was professional and courteous        Positive
5  Reached destination but took longer than expected         Neutral
6    Driver cancelled last minute, very inconvenient        Negative
7  Driver followed the route perfectly, very sati...        Positive
8                 Driver was rude and unprofessional        Negative
9            Driver took wrong route and overcharged        Negative


In [36]:
df.columns

Index(['ride_id', 'timestamp', 'pickup_zone', 'drop_zone', 'pickup_lat',
       'pickup_long', 'drop_lat', 'drop_long', 'driver_id', 'customer_id',
       'fare_price', 'surge_multiplier', 'driver_rating', 'customer_rating',
       'estimated_eta_min', 'actual_eta_min', 'ride_status', 'traffic_level',
       'weather', 'driver_active', 'feedback_text', 'year', 'month', 'day',
       'hour', 'minute', 'day_of_week', 'is_weekend', 'eta_error',
       'pickup_cluster', 'sentiment_label'],
      dtype='object')

In [ ]:
#8️⃣ Feedback Intelligence

In [37]:
from groq import Groq
import pandas as pd
import time

client = Groq(api_key="Your Key")

def analyze_feedback(feedback_text):
    prompt = f"""You are a ride-hailing feedback analyzer for RideFlow.

Analyze this customer feedback and respond ONLY in this exact format:
Sentiment: [Positive/Neutral/Negative]
Issue: [Driver Behavior/Vehicle Condition/Cancellation/Delay/Route & Fare/General/None]
Score: [1-10]

Feedback: "{feedback_text}"

Rules:
- Sentiment must be exactly: Positive, Neutral, or Negative
- Issue must be exactly one of the listed categories
- Score 1=very negative, 10=very positive
- No extra text, just the 3 lines"""

    response = client.chat.completions.create(
        model    = "llama-3.3-70b-versatile",
        messages = [{"role": "user", "content": prompt}],
        max_tokens = 60,
        temperature = 0
    )
    return response.choices[0].message.content.strip()

# Test on 5 samples first
test_feedbacks = [
    "Driver was rude and unprofessional",
    "Excellent service, will use again",
    "Neither good nor bad, just a normal ride",
    "Driver cancelled last minute, very inconvenient",
    "Vehicle was very dirty and uncomfortable"
]

print("=== Testing Groq Feedback Analyzer ===\n")
for fb in test_feedbacks:
    result = analyze_feedback(fb)
    print(f"Feedback: '{fb}'")
    print(f"Result:\n{result}")
    print("-" * 50)

=== Testing Groq Feedback Analyzer ===

Feedback: 'Driver was rude and unprofessional'
Result:
Sentiment: Negative
Issue: Driver Behavior
Score: 1
--------------------------------------------------
Feedback: 'Excellent service, will use again'
Result:
Sentiment: Positive
Issue: None
Score: 10
--------------------------------------------------
Feedback: 'Neither good nor bad, just a normal ride'
Result:
Sentiment: Neutral
Issue: None
Score: 5
--------------------------------------------------
Feedback: 'Driver cancelled last minute, very inconvenient'
Result:
Sentiment: Negative
Issue: Cancellation
Score: 2
--------------------------------------------------
Feedback: 'Vehicle was very dirty and uncomfortable'
Result:
Sentiment: Negative
Issue: Vehicle Condition
Score: 2
--------------------------------------------------


In [ ]:
#as only 19 different feedbacks are there . so mapped so accordingly by calling the API only 19 times

In [38]:
def parse_response(response_text):
    lines = response_text.strip().split('\n')
    result = {'sentiment': 'Neutral', 'issue': 'General', 'score': 5}
    
    for line in lines:
        if line.startswith('Sentiment:'):
            result['sentiment'] = line.split(':')[1].strip()
        elif line.startswith('Issue:'):
            result['issue'] = line.split(':')[1].strip()
        elif line.startswith('Score:'):
            try:
                result['score'] = int(line.split(':')[1].strip())
            except:
                result['score'] = 5
    return result

# Get unique feedbacks only
unique_feedbacks = df['feedback_text'].unique()
print(f"Unique feedbacks to analyze: {len(unique_feedbacks)}")

# Analyze all unique feedbacks
feedback_results = {}

for i, feedback in enumerate(unique_feedbacks):
    try:
        raw     = analyze_feedback(feedback)
        parsed  = parse_response(raw)
        feedback_results[feedback] = parsed
        print(f"[{i+1}/{len(unique_feedbacks)}] '{feedback[:50]}' → {parsed['sentiment']} | {parsed['issue']} | {parsed['score']}")
        time.sleep(0.5)  # avoid rate limiting
    except Exception as e:
        print(f"Error on '{feedback}': {e}")
        feedback_results[feedback] = {'sentiment': 'Neutral', 'issue': 'General', 'score': 5}

print("\n✅ All unique feedbacks analyzed!")

Unique feedbacks to analyze: 19
[1/19] 'AC was not working, driver was unhelpful' → Negative | Vehicle Condition | 2
[2/19] 'Ride was okay, nothing special' → Neutral | General | 5
[3/19] 'Driver took wrong route and overcharged' → Negative | Route & Fare | 2
[4/19] 'Driver was rude and unprofessional' → Negative | Driver Behavior | 1
[5/19] 'Driver was professional and courteous' → Positive | Driver Behavior | 10
[6/19] 'Reached destination but took longer than expected' → Negative | Route & Fare | 4
[7/19] 'Driver cancelled last minute, very inconvenient' → Negative | Cancellation | 1
[8/19] 'Driver followed the route perfectly, very satisfie' → Positive | Route & Fare | 10
[9/19] 'Very clean vehicle and friendly driver' → Positive | Vehicle Condition | 10
[10/19] 'Smooth ride, reached on time, great experience' → Positive | None | 10
[11/19] 'Comfortable ride, no issues at all' → Positive | None | 10
[12/19] 'Very bad experience, driver was aggressive' → Negative | Driver Behavior |

In [39]:
# Map results back to all 50,000 rows
df['groq_sentiment'] = df['feedback_text'].map(
    lambda x: feedback_results.get(x, {}).get('sentiment', 'Neutral')
)

df['groq_issue'] = df['feedback_text'].map(
    lambda x: feedback_results.get(x, {}).get('issue', 'General')
)

df['groq_score'] = df['feedback_text'].map(
    lambda x: feedback_results.get(x, {}).get('score', 5)
)

print("=== Sentiment Distribution ===")
print(df['groq_sentiment'].value_counts())

print("\n=== Issue Distribution ===")
print(df['groq_issue'].value_counts())

print("\n=== Score Distribution ===")
print(df['groq_score'].describe())

print("\n=== Sample Output ===")
print(df[['feedback_text','groq_sentiment','groq_issue','groq_score']].head(10))

=== Sentiment Distribution ===
groq_sentiment
Positive    25071
Negative    17019
Neutral      7910
Name: count, dtype: int64

=== Issue Distribution ===
groq_issue
Driver Behavior      13403
None                 12740
Vehicle Condition     9919
Route & Fare          7764
Cancellation          2120
Delay                 2046
General               2008
Name: count, dtype: int64

=== Score Distribution ===
count    50000.000000
mean         6.437160
std          3.757187
min          1.000000
25%          2.000000
50%         10.000000
75%         10.000000
max         10.000000
Name: groq_score, dtype: float64

=== Sample Output ===
                                       feedback_text groq_sentiment  \
0           AC was not working, driver was unhelpful       Negative   
1                     Ride was okay, nothing special        Neutral   
2            Driver took wrong route and overcharged       Negative   
3                 Driver was rude and unprofessional       Negative   
4    

In [40]:
df.columns

Index(['ride_id', 'timestamp', 'pickup_zone', 'drop_zone', 'pickup_lat',
       'pickup_long', 'drop_lat', 'drop_long', 'driver_id', 'customer_id',
       'fare_price', 'surge_multiplier', 'driver_rating', 'customer_rating',
       'estimated_eta_min', 'actual_eta_min', 'ride_status', 'traffic_level',
       'weather', 'driver_active', 'feedback_text', 'year', 'month', 'day',
       'hour', 'minute', 'day_of_week', 'is_weekend', 'eta_error',
       'pickup_cluster', 'sentiment_label', 'groq_sentiment', 'groq_issue',
       'groq_score'],
      dtype='object')

In [ ]:
#checked the metrics 

In [41]:
from sklearn.metrics import f1_score, classification_report

# Compare groq vs our simulated labels
print("=== F1 Score (Groq vs Simulated Labels) ===")
print(classification_report(
    df['sentiment_label'],
    df['groq_sentiment'],
    target_names=['Negative','Neutral','Positive']
))

f1 = f1_score(df['sentiment_label'], df['groq_sentiment'], average='weighted')
print(f"Weighted F1: {f1:.4f}")
if f1 >= 0.80:
    print("✅ Benchmark achieved! F1 > 0.80")
else:
    print("⚠️ Below benchmark")

=== F1 Score (Groq vs Simulated Labels) ===
              precision    recall  f1-score   support

    Negative       0.88      1.00      0.94     15010
     Neutral       1.00      0.80      0.89      9919
    Positive       1.00      1.00      1.00     25071

    accuracy                           0.96     50000
   macro avg       0.96      0.93      0.94     50000
weighted avg       0.96      0.96      0.96     50000

Weighted F1: 0.9588
✅ Benchmark achieved! F1 > 0.80


In [ ]:
#9️⃣ AI Ride Matching Assistant using Groq LLM

In [44]:
#creating a driver profile

from sklearn.preprocessing import MinMaxScaler


np.random.seed(42)
df['driver_id'] = np.random.randint(1, 201, size=len(df))
df['driver_active'] = (df['driver_active'] > 0.5).astype(int)

# Build driver profile
driver_profile = df.groupby('driver_id').agg(
    avg_rating        = ('driver_rating',  'mean'),
    avg_eta           = ('actual_eta_min', 'mean'),
    total_rides       = ('ride_status',    'count'),
    cancelled_rides   = ('ride_status',    lambda x: (x == 'cancelled').sum()),
    active_ratio      = ('driver_active',  'mean'),
    avg_fare          = ('fare_price',     'mean')
).reset_index()

driver_profile['cancellation_rate'] = (
    driver_profile['cancelled_rides'] / driver_profile['total_rides']
).round(3)

driver_profile['avg_rating'] = driver_profile['avg_rating'].round(3)
driver_profile['avg_eta']    = driver_profile['avg_eta'].round(2)
driver_profile['avg_fare']   = driver_profile['avg_fare'].round(2)

# Match score
scaler = MinMaxScaler()
driver_profile['rating_score'] = scaler.fit_transform(driver_profile[['avg_rating']])
driver_profile['eta_score']    = 1 - scaler.fit_transform(driver_profile[['avg_eta']])
driver_profile['cancel_score'] = 1 - scaler.fit_transform(driver_profile[['cancellation_rate']])

driver_profile['match_score'] = (
    0.4 * driver_profile['rating_score'] +
    0.4 * driver_profile['eta_score']    +
    0.2 * driver_profile['cancel_score']
).round(4)

print(f"✅ Driver profile built: {len(driver_profile)} drivers")
print("\nTop 5 Drivers:")
print(driver_profile.nlargest(5, 'match_score')[
    ['driver_id','avg_rating','avg_eta','cancellation_rate','match_score']
])

✅ Driver profile built: 200 drivers

Top 5 Drivers:
     driver_id  avg_rating  avg_eta  cancellation_rate  match_score
126        127       4.118    13.49              0.241       0.7579
135        136       4.057    13.55              0.152       0.7540
42          43       4.061    13.39              0.179       0.7521
76          77       4.087    13.40              0.229       0.7326
84          85       4.076    12.96              0.281       0.7167


In [46]:
import json
from groq import Groq

client = Groq(api_key="Your Key")

def groq_recommend_driver(top_drivers_df, customer_location="Chennai Central", 
                           hour=18, traffic="high"):
    
    drivers_info = top_drivers_df.head(3)[
        ['driver_id','avg_rating','avg_eta','cancellation_rate','match_score']
    ].to_dict(orient='records')

    prompt = f"""You are an AI Ride Matching Assistant for RideFlow, a ride-hailing platform in Chennai.

Current Context:
- Customer Location: {customer_location}
- Time: {hour}:00 hrs
- Traffic: {traffic}
- City: Chennai, Tamil Nadu

Top 3 Available Drivers:
{json.dumps(drivers_info, indent=2)}

Field Descriptions:
- avg_rating: Driver rating out of 5 (higher = better)
- avg_eta: Average arrival time in minutes (lower = better)
- cancellation_rate: Fraction of rides cancelled (lower = better)
- match_score: Overall AI match score 0-1 (higher = better)

Provide your recommendation in EXACTLY this format:

Recommended Driver ID: [ID]
Match Score: [score]
Reason: [2-3 sentences explaining why this driver is best — mention ETA, rating and cancellation risk specifically]
Alternative: Driver [ID] is also a strong option because [one sentence]
Risk Flag: [If cancellation_rate > 0.25 write a warning, else write None]
Confidence: [High/Medium/Low based on how close the scores are]
"""

    response = client.chat.completions.create(
        model    = "llama-3.3-70b-versatile",
        messages = [{"role": "user", "content": prompt}],
        max_tokens  = 300,
        temperature = 0.3
    )
    
    return response.choices[0].message.content.strip()

# Test it
top5 = driver_profile.nlargest(5, 'match_score')
result = groq_recommend_driver(top5, 
                                customer_location="T Nagar, Chennai",
                                hour=18, 
                                traffic="high")
print(result)

Recommended Driver ID: 127
Match Score: 0.7579
Reason: Driver 127 has the highest match score and a relatively low average ETA of 13.49 minutes, which is impressive given the high traffic conditions. Additionally, their average rating of 4.118 out of 5 indicates a high level of customer satisfaction, and their cancellation rate of 0.241 is manageable. This combination makes them the most suitable choice for the ride.
Alternative: Driver 136 is also a strong option because they have a slightly lower cancellation rate and a similar average ETA.
Risk Flag: None
Confidence: Medium


In [47]:
def test_scenarios():
    scenarios = [
        {"location": "Chennai Airport",    "hour": 6,  "traffic": "low"},
        {"location": "T Nagar",            "hour": 18, "traffic": "high"},
        {"location": "Anna Nagar",         "hour": 13, "traffic": "medium"},
        {"location": "Tambaram",           "hour": 23, "traffic": "low"},
    ]

    for s in scenarios:
        print(f"\n{'='*60}")
        print(f"📍 {s['location']} | 🕐 {s['hour']}:00 | 🚦 {s['traffic'].upper()}")
        print('='*60)
        result = groq_recommend_driver(
            top5,
            customer_location = s['location'],
            hour              = s['hour'],
            traffic           = s['traffic']
        )
        print(result)

test_scenarios()


📍 Chennai Airport | 🕐 6:00 | 🚦 LOW
Recommended Driver ID: 127
Match Score: 0.7579
Reason: Driver 127 has the highest match score, with a relatively low average ETA of 13.49 minutes and a high average rating of 4.118, indicating a reliable and satisfactory ride experience. Although the cancellation rate is slightly higher at 0.241, it is still below the threshold, making this driver a preferable choice. The balance of these factors contributes to the highest overall match score.
Alternative: Driver 136 is also a strong option because it has a similar average ETA and a slightly lower cancellation rate, making it a viable alternative.
Risk Flag: None
Confidence: High

📍 T Nagar | 🕐 18:00 | 🚦 HIGH
Recommended Driver ID: 127
Match Score: 0.7579
Reason: Driver 127 is the top recommendation due to their high average rating of 4.118, relatively low average ETA of 13.49 minutes, and a manageable cancellation rate of 0.241. Although the cancellation rate is slightly higher than the other option

In [48]:
import joblib

joblib.dump(driver_profile, 'driver_profile.pkl')
print("✅ Driver profile saved!")
print("Top 3 drivers:", driver_profile.nlargest(3,'match_score')['driver_id'].tolist())

✅ Driver profile saved!
Top 3 drivers: [127, 136, 43]


In [49]:
# 🔟 AI Chatbot with Groq LLM

In [50]:
from groq import Groq

client = Groq(api_key="Your key")

def rideflow_chatbot(user_message, language="English", role="customer"):
    
    system_prompt = f"""You are RideFlow AI Assistant, a helpful support agent for a ride-hailing platform in Chennai, India.

You are talking to a {role}.

Your responsibilities:
- Customer queries: booking, ride status, refunds, complaints, ETA
- Driver queries: earnings, navigation, cancellations, ratings

Rules:
- Always respond in {language} language only
- Be concise — max 3 sentences
- Be empathetic and professional
- For refunds mention 3-5 business days
- For ETA give realistic Chennai traffic estimates
- Never make up ride details you don't know

Current context:
- City: Chennai, Tamil Nadu
- Platform: RideFlow
- Supported zones: T Nagar, Anna Nagar, Tambaram, Velachery, Adyar, Porur, OMR
"""

    response = client.chat.completions.create(
        model    = "llama-3.3-70b-versatile",
        messages = [
            {"role": "system",  "content": system_prompt},
            {"role": "user",    "content": user_message}
        ],
        max_tokens  = 150,
        temperature = 0.5
    )
    
    return response.choices[0].message.content.strip()

# Test English
print("=== English ===")
print(rideflow_chatbot("Where is my driver?", language="English", role="customer"))

print("\n=== Tamil ===")
print(rideflow_chatbot("என் டிரைவர் எங்கே?", language="Tamil", role="customer"))

print("\n=== Hindi ===")
print(rideflow_chatbot("मेरा ड्राइवर कहाँ है?", language="Hindi", role="customer"))

=== English ===
I've checked on the status of your ride, and it seems your driver is en route to your location. Due to Chennai's traffic, the estimated time of arrival is around 15-20 minutes. Please wait for a few more minutes, and if you need further assistance, feel free to ask.

=== Tamil ===
வணக்கம், தயவு செய்து உங்கள் ரைட் ஆர்டர் எண்ணை எங்களுக்கு தெரிவிக்கவும், நாங்கள் உங்கள் டிரைவரின் நிலைமையை பார்ப

=== Hindi ===
आपके ड्राइवर की वर्तमान स्थिति की जांच करने के लिए मैं आपकी राइड की जानकारी देखने की कोशिश कर रहा हूँ। चेन्नई के भीड़भाड़ वाले यातायात को देखते हुए, आपका ड्राइवर लगभग 15-20 मिनट में आपके पिकअप स्थान पर पहुंच सकता है। कृपया अपने राइड की स्थिति के लिए राइडफ्लो ऐप की जांच करें।


In [52]:
#building full chatbot with conversation memory

In [51]:
def rideflow_chatbot_with_memory(conversation_history, user_message, 
                                  language="English", role="customer"):
    
    system_prompt = f"""You are RideFlow AI Assistant, a helpful support agent for a ride-hailing platform in Chennai, India.

You are talking to a {role}.

Your responsibilities:
Customer queries: booking, ride status, refunds, complaints, ETA
Driver queries: earnings, navigation, cancellations, ratings

Rules:
- Always respond in {language} only
- Be concise — max 3 sentences
- Be empathetic and professional
- Refunds take 3-5 business days
- For ETA give realistic Chennai traffic estimates
- Never make up ride details you don't know

Platform context:
- City: Chennai, Tamil Nadu
- Platform: RideFlow
- Zones: T Nagar, Anna Nagar, Tambaram, Velachery, Adyar, Porur, OMR"""

    # Build messages with full history
    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(conversation_history)
    messages.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model       = "llama-3.3-70b-versatile",
        messages    = messages,
        max_tokens  = 150,
        temperature = 0.5
    )

    reply = response.choices[0].message.content.strip()

    # Update history
    conversation_history.append({"role": "user",      "content": user_message})
    conversation_history.append({"role": "assistant", "content": reply})

    return reply, conversation_history


# ── Test multi-turn conversation ──────────────────────────────
print("=== Multi-turn Customer Conversation ===\n")

history = []

queries = [
    "Hi, I booked a ride but my driver hasn't arrived yet",
    "It's been 20 minutes already!",
    "Can I get a refund if I cancel now?",
    "Ok I cancelled. When will I get my refund?"
]

for query in queries:
    print(f"Customer: {query}")
    reply, history = rideflow_chatbot_with_memory(
        history, query, language="English", role="customer"
    )
    print(f"RideFlow: {reply}\n")

=== Multi-turn Customer Conversation ===

Customer: Hi, I booked a ride but my driver hasn't arrived yet
RideFlow: I apologize for the inconvenience, and I'm here to help. Can you please provide me with your booking ID or the pickup location so I can check on the status of your ride? I'll do my best to assist you and provide an estimated arrival time, considering Chennai's traffic.

Customer: It's been 20 minutes already!
RideFlow: I understand your frustration, and I apologize for the delay. I've checked on the status, but I don't have real-time information on the driver's location. If you'd like, I can try to reach out to the driver or provide an alternative solution to get you to your destination as soon as possible.

Customer: Can I get a refund if I cancel now?
RideFlow: If you cancel your ride now, you will be eligible for a refund, which will be processed within 3-5 business days. Please note that cancellation fees may apply, depending on our platform's policies. Would you like 

In [53]:
print("=== Driver Conversation ===\n")

driver_history = []

driver_queries = [
    "How much did I earn today?",
    "Why was my rating reduced?",
    "How do I navigate to OMR from Tambaram?"
]

for query in driver_queries:
    print(f"Driver: {query}")
    reply, driver_history = rideflow_chatbot_with_memory(
        driver_history, query, language="English", role="driver"
    )
    print(f"RideFlow: {reply}\n")

=== Driver Conversation ===

Driver: How much did I earn today?
RideFlow: Your total earnings for the day are ₹2,500. You've completed 15 trips so far, with the highest earnings from the OMR zone. Would you like me to break down your earnings by zone or trip type?

Driver: Why was my rating reduced?
RideFlow: Your rating was reduced due to a passenger complaint about a route deviation during a trip from T Nagar to Anna Nagar. I can provide more details about the specific feedback if you'd like to know.

Driver: How do I navigate to OMR from Tambaram?
RideFlow: To navigate to OMR from Tambaram, take the GST Road towards Chennai and then merge onto the OMR highway. You can use the in-app navigation or follow the GPS route suggested by RideFlow for the most efficient route.



In [54]:
import joblib
import json

# 1. Save driver profile
joblib.dump(driver_profile, 'driver_profile.pkl')

# 2. Save feedback results
with open('feedback_results.json', 'w') as f:
    json.dump(feedback_results, f)

# 3. Save final dataframe
df.to_csv("rideflow_module3_groq.csv", index=False)

print(" All saved!")
print("Files:")
print("  - driver_profile.pkl")
print("  - feedback_results.json")
print("  - rideflow_module3_groq.csv")

 All saved!
Files:
  - driver_profile.pkl
  - feedback_results.json
  - rideflow_module3_groq.csv


In [55]:
import joblib

demand_model   = joblib.load('demand_prediction_model.pkl')
supply_model   = joblib.load('driver_supply_model.pkl')
cancel_model   = joblib.load('cancellation_model.pkl')
features_cancel= joblib.load('features_cancel.pkl')

print("=== Demand Model Features ===")
print(demand_model.get_booster().feature_names)

print("\n=== Supply Model Features ===")
print(supply_model.feature_names_in_)

print("\n=== Cancel Model Features ===")
print(cancel_model.feature_names_in_)

print("\n=== Features Cancel PKL ===")
print(features_cancel)

=== Demand Model Features ===
['pickup_cluster', 'hour', 'lag_1h', 'lag_2h', 'rolling_3h']

=== Supply Model Features ===
['hour' 'day_of_week' 'is_weekend' 'cluster_avg_supply' 'supply_lag_1h'
 'supply_lag_2h' 'supply_trend' 'rolling_3h' 'traffic_level_enc']

=== Cancel Model Features ===
['hour' 'day_of_week' 'is_weekend' 'traffic_level_enc' 'gap'
 'surge_multiplier' 'pickup_cluster']

=== Features Cancel PKL ===
['estimated_eta_min', 'fare_price', 'surge_multiplier', 'driver_rating', 'traffic_level_enc', 'pickup_cluster']
